# GeoMaster on Google Colab
公開GitHubリポジトリから最新版を取得し、HTMLとJavaScriptをセル出力へ直接埋め込んで実行します。ポート公開やGitHubトークンは不要です。

In [ ]:
from pathlib import Path
import os, shutil, subprocess

repo_dir = Path('/content/GeoMaster')
os.chdir('/content')
if repo_dir.exists():
    shutil.rmtree(repo_dir)

result = subprocess.run([
    'git', 'clone', '--depth', '1',
    'https://github.com/TomTomYoung/GeoMaster.git',
    str(repo_dir)
], text=True, capture_output=True)

print(result.stdout, end='')
if result.returncode != 0:
    print(result.stderr, end='')
    raise RuntimeError(f'git clone failed: exit {result.returncode}')

index_file = repo_dir / 'index.html'
assert index_file.is_file(), f'index.html がありません: {index_file}'
print('Repository:', repo_dir)
print('Entry file:', index_file)


In [ ]:
from IPython.display import HTML, display
import re

html = (repo_dir / 'index.html').read_text(encoding='utf-8')

# ES Modulesを1本の通常スクリプトへ結合する。
# Colab出力フレームから src/app.js 等を取得させないため、外部HTTP要求は発生しない。
parts = []
for relative_path in [
    'src/geometry.js',
    'src/renderer.js',
    'src/interaction.js',
    'src/app.js',
]:
    source = (repo_dir / relative_path).read_text(encoding='utf-8')
    source = re.sub(r'^\s*import\s+.*?;\s*$', '', source, flags=re.MULTILINE)
    source = re.sub(r'\bexport\s+(?=(?:const|let|var|function|class)\b)', '', source)
    parts.append(f'// ---- {relative_path} ----\n{source}')

bundle = '\n\n'.join(parts)
inline_script = '<script>\n' + bundle.replace('</script>', '<\\/script>') + '\n</script>'

html, replaced = re.subn(
    r'<script\s+type=["\']module["\']\s+src=["\']src/app\.js["\']\s*></script>',
    lambda _: inline_script,
    html,
    count=1,
)
assert replaced == 1, 'index.html の app.js 読込みタグを置換できませんでした'

display(HTML(html))
